# 04 · Revisión con Martin

## Qué hicimos

**Objetivo:** llenar la tabla de Martin (una fila por individuo y lado) para los registros de *Sepia officinalis* de iNaturalist en el Mar del Norte, con un flujo reproducible: scripts fijos, prompts versionados y validación contra anotación manual.

**1. Datos.** Se descargó cada sector a Drive **una sola vez** (copia cruda de la API, fotos originales y versiones reducidas). Coordenadas directamente desde la API, sin modificar. Dos corridas de prueba:
- **A1** (`pilot_a1`): rectángulo aproximado del sector A1.
- **A2** (`a2_test`): rectángulo aproximado del sector A2, sin volver a bajar los registros que ya estaban en A1.

Los rectángulos son aproximados; los límites exactos de los sectores (PPT de Martin) quedan para el final.

**2. Anotación manual** (interfaz propia en Colab): una fila por **foto e individuo**, con letra (a, b, c…), lado, calidad, erosión, picoteo, epibiontes, certeza, notas, recuadro y duplicados.
- **A1** se anotó **a ciegas** (sin ver los modelos): es la referencia para medir.
- **A2** se anotó de forma **asistida** (primero se vieron los modelos y se eligió el valor correcto, luego se revisó y completó): más rápido, pero no sirve para medir los modelos.

**3. Modelos** (Gemini `gemini-3.5-flash-lite`, temperatura 0):
- **Triaje** por registro: estado y número de huesos.
- **Rasgos por foto** con 4 versiones de prompt: lado, erosión y calidad cuando hay 1 hueso visible.
  - v8: dorsal por crestas rectas a lo largo;
  - v12: v8 + "si dudas, `unclear`" + con erosión 3 el lado queda `unclear`;
  - v13: v12 sin "poroso/tiza" como señal ventral (causaba falsos ventrales) y dorsal "más liso y duro" (guía de Martin);
  - v14: v13 + calidad descrita (qué tapa el hueso, foco) y decidida por una regla fija.
- **Moda** de los 3 mejores: el valor que dan al menos 2 de los 3.

**4. Evaluación:** matriz de confusión, exactitud, cobertura, precisión, recall y F1, por separado en A1 y A2.

**5. Entrega:** Excel generado por script con las hojas de Martin (`legend`, `map`, `identification`, `A1`, `A2`) más `traceability` y `excluded`. **Solo anotación manual**: nada de los modelos va al Excel.

## Contenido

0. Datos de las dos corridas
1. Interfaz de anotación
2. Resultados: métricas y matrices (A1 y A2 por separado), con explicación de cada métrica
3. Todas las fotos (A1 + A2)
4. Fotos donde al menos un prompt difiere de la anotación manual
5. Fotos donde los 4 prompts coinciden entre sí y difieren de la anotación manual
6. Fotos con notas para preguntar a Martin
7. Corregir esas fotos durante la reunión
8. Preguntas para Martin
9. Excel de entrega

En las fotos se dibuja el recuadro y la letra de cada individuo (sin recuadro: la letra aparece bajo la foto con "no box"). En las vistas 4 y 5, `unclear` cuenta como diferencia (gris). En las métricas, `unclear` del modelo es una abstención.

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
!pip install -q jupyter-bbox-widget          # recuadros en la interfaz de anotacion

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'

In [ ]:
# Corridas (cada una en su carpeta de Drive). A1: anotacion a ciegas. A2: anotacion asistida.
cfg_a1 = sepia.load_config('configs/pilot_a1.yaml')
cfg_a2 = sepia.load_config('configs/a2_test.yaml')
RUNS = [cfg_a1, cfg_a2]

In [ ]:
from sepia.annotate import AnnotationApp, load_events, done_photos
from sepia.disagree import (DisagreementApp, classification_metrics, plot_confusion,
                            count_agreement, rank_versions, ask_martin_photos)

## 0. Datos de las dos corridas

In [ ]:
for name, c in (('A1 (a ciegas)', cfg_a1), ('A2 (asistida)', cfg_a2)):
    obs, pho = sepia.load_tables(c)
    ev = load_events(c)
    ind = ev['individual']
    done = done_photos(ev)
    print(f"{name}: {len(obs)} registros · {len(pho)} fotos · {len(done)} fotos anotadas · "
          f"{len(ind)} filas foto × individuo · "
          f"{ind.groupby('observation_id')['individual'].nunique().sum() if len(ind) else 0} huesos distintos")
    for v in VERSIONS:
        print(f"   {v}: {len(sepia.load_photo_traits(c, v))} fotos con resultado")

## 1. Interfaz de anotación

Cómo se anota (A1). **Guarda de verdad**: para mostrarla sin cambiar nada, navegar sin apretar *Save*.

In [ ]:
AnnotationApp(cfg_a1, ANNOTATOR).show()

## 2. Resultados de los 4 prompts y de la moda

Sobre las mismas fotos para todos. Lado, erosión y calidad: solo fotos con 1 individuo anotado. Los 3 mejores para la moda se eligen con el ranking de A1. Ojo: elegirlos y evaluarlos con las mismas fotos da una cifra algo optimista; falta confirmarla con fotos nuevas anotadas a ciegas.

In [ ]:
ranking, _ = rank_versions(cfg_a1, VERSIONS, n_top=3, save=False)
TOP3 = list(ranking.index[:3])
print('3 mejores (A1):', TOP3)

### Qué mide cada métrica

Con dos clases (por ejemplo dorsal y ventral) y mirando la clase **dorsal**:

| | el modelo dice dorsal | el modelo dice ventral o `unclear` |
|---|---|---|
| **anotado dorsal** | VP (verdadero positivo) | FN (falso negativo) |
| **anotado ventral** | FP (falso positivo) | VN (verdadero negativo) |

- **Exactitud (accuracy)** = fotos acertadas / todas las fotos. Un `unclear` del modelo cuenta como error. **accuracy_when_decided** la calcula solo donde el modelo decidió.
- **Precisión** de dorsal = VP / (VP + FP): cuando el modelo dice "dorsal", ¿cuántas veces es dorsal? Baja = falsas alarmas.
- **Recall** de dorsal = VP / (VP + FN): de los dorsales anotados, ¿cuántos encontró? Baja = se le escapan.
- **F1** = 2 · precisión · recall / (precisión + recall): solo es alta si las dos son altas.
- **Cobertura**: fracción de fotos donde el modelo decidió (no `unclear`).
- **macro F1** y **exactitud balanceada** (promedio de los recall): cada clase pesa lo mismo.

Al final de la sección 2a hay un ejemplo práctico con los números de la moda.

### 2a. A1: anotación a ciegas (referencia para medir los prompts)

In [ ]:
count_agreement(cfg_a1, VERSIONS, mode_of=TOP3);
summary_a1, per_class_a1, mats_a1 = classification_metrics(cfg_a1, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a1, 'side');           # fila 'unclear' = anotado unclear (fuera de las metricas)
plot_confusion(mats_a1, 'erosion');
plot_confusion(mats_a1, 'photo_quality');

#### Ejemplo práctico: las métricas del lado, paso a paso (moda, A1)

In [ ]:
cm = mats_a1[('mode', 'side')]
print(cm, '\n')
rows = [c for c in ('dorsal', 'ventral') if c in cm.index]
n = int(cm.loc[rows].values.sum())                       # fotos con lado anotado dorsal/ventral
hits = sum(int(cm.loc[c, c]) for c in rows if c in cm.columns)
decided = int(cm.loc[rows, [c for c in rows if c in cm.columns]].values.sum())
print(f"Exactitud = aciertos / fotos = {hits} / {n} = {hits / n:.2f}   ('unclear' cuenta como error)")
print(f"Exactitud al decidir = {hits} / {decided} = {hits / decided:.2f}")
print(f"Cobertura = decididas / fotos = {decided} / {n} = {decided / n:.2f}\n")
for c in rows:
    vp = int(cm.loc[c, c]) if c in cm.columns else 0
    fp = int(cm.loc[[r for r in rows if r != c], c].sum()) if c in cm.columns else 0
    fn = int(cm.loc[c].sum()) - vp
    prec = vp / (vp + fp) if vp + fp else float('nan')
    rec = vp / (vp + fn) if vp + fn else float('nan')
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float('nan')
    print(f"{c}: VP={vp} FP={fp} FN={fn}")
    print(f"   precision = VP/(VP+FP) = {vp}/{vp + fp} = {prec:.2f}")
    print(f"   recall    = VP/(VP+FN) = {vp}/{vp + fn} = {rec:.2f}")
    print(f"   F1        = 2·P·R/(P+R) = {f1:.2f}")

### 2b. A2: anotación asistida (solo referencia)

El acuerdo con los prompts sale **inflado**: la anotación partió de sus respuestas.

In [ ]:
count_agreement(cfg_a2, VERSIONS, mode_of=TOP3);
summary_a2, per_class_a2, mats_a2 = classification_metrics(cfg_a2, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a2, 'side');
plot_confusion(mats_a2, 'erosion');
plot_confusion(mats_a2, 'photo_quality');

## 3. Todas las fotos, una a una (A1 + A2)

Foto grande con recuadros y letras, miniaturas del registro (clic = ver en grande), tabla con los 4 prompts, la moda y la anotación manual, evidencia de cada prompt y resumen del registro. Arriba: la corrida y si la anotación fue **a ciegas** (verde) o **asistida** (naranjo). *Run* filtra por corrida, *Field* por atributo y *Go to ID* salta a un registro.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='all', read_only=True, mode_of=TOP3).show()

## 4. Al menos un prompt difiere de la anotación manual

Rojo = el prompt decidió otra cosa; gris = el prompt dijo `unclear`.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='any_model_differs', read_only=True, mode_of=TOP3).show()

## 5. Los 4 prompts coinciden entre sí y difieren de la anotación manual

Candidatas más fuertes a error de anotación, o a un criterio distinto entre la persona y los prompts.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='all_models_agree_differ', read_only=True, mode_of=TOP3).show()

## 6. Fotos con notas para Martin

Fotos donde alguna nota menciona a Martin (martin, Martín, marthin). La nota aparece destacada arriba.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='ask_martin', read_only=True, mode_of=TOP3).show()

## 7. Corregir esas fotos durante la reunión

Las mismas fotos de la sección 6, en la interfaz de anotación. **Guarda de verdad**: cada *Save* agrega una línea y vale la última. *Prev* / *Next* solo pasan por las fotos con nota.

In [ ]:
ids_a1 = ask_martin_photos(cfg_a1)
print(f"A1: {len(ids_a1)} fotos con nota para Martin")
if ids_a1:
    AnnotationApp(cfg_a1, ANNOTATOR, photo_ids=ids_a1).show()

In [ ]:
ids_a2 = ask_martin_photos(cfg_a2)
print(f"A2: {len(ids_a2)} fotos con nota para Martin")
if ids_a2:
    AnnotationApp(cfg_a2, ANNOTATOR, photo_ids=ids_a2).show()

## 8. Preguntas para Martin

**Criterios de anotación**
1. **Lado:** ¿las crestas rectas a lo largo son una señal del lado dorsal en *S. officinalis*? En la diapositiva 9 de la guía, ¿la figura con estrías en arco es la vista ventral? ¿El surco central a lo largo se ve en huesos varados?
2. **Calidad:** ¿le sirven los umbrales? high = nada encima o un par de granos (≥ ~95 % visible); medium = varios granos, pasto, algas o dedos tapan parte (~5-60 %); low = gran parte tapada (> ~60 %) o borrosa.
3. **Erosión y lado:** con erosión 3 el lado queda `unclear` (el acierto del lado cae con la erosión). ¿Le parece bien?
4. **Fragmentos:** regla provisoria "medio hueso o más cuenta como hueso". ¿Está bien?
5. **Picoteo:** ¿qué significa "(yes)"? ¿Un dorsal sin marcas es "no" o "no se puede evaluar"?
6. **Certeza:** ¿por fila o por variable? ¿Se excluyen de la evaluación las fotos con duda?

**Datos y formato**
7. **Sectores:** límites exactos (PPT, diapositivas 5-12), distancia máxima a la costa, rango de fechas, ¿solo *Research Grade*?
8. **Excel:** una fila por **foto × individuo** (el mismo hueso en dos fotos = dos filas); el lado en **una columna** (`side`) en vez de dos; columnas nuevas `photo ID`, `individual` y duplicados. `private_latitude` / `private_longitude` quedan vacías (la API pública no las entrega).
9. **Exclusiones:** ¿qué motivos excluyen un registro y cuáles solo lo marcan?
10. **Duplicados entre usuarios:** mismo día y ≤ 2 km como candidatos, confirmados por una persona. ¿Le sirve?

**Evaluación**
11. **Umbral de acierto** para aceptar el lado del modelo (por ejemplo ≥ 90 % cuando decide) y cuánto `unclear` se acepta.
12. **Acuerdo entre personas:** ¿podría anotar a ciegas unas 50 fotos? Así se calcula el kappa de Cohen persona-persona, que es el techo para el modelo.
13. ¿Otros anotadores? ¿Plazo? ¿Publicar el código?

## 9. Excel de entrega

Un solo Excel con A1 y A2 (las corridas siguen separadas en Drive). Hojas: `legend`, `map`, `identification` (una fila por foto × individuo), `A1` y `A2` (una fila por registro, columnas de iNaturalist como en la tabla de Martin + resumen de la anotación), `traceability` y `excluded`. **Solo anotación manual.** Se guarda en `MyDrive/sepia-inaturalist/exports/`.

In [ ]:
out = sepia.build_excel(RUNS, 'sepia_A1_A2.xlsx')
print(out)